# SIE Quickstart

Three functions - `encode`, `score`, `extract` - cover the full API.

## 1. Start the server

Run the SIE server with Docker before using this notebook:

```bash
# CPU
docker run -p 8080:8080 ghcr.io/superlinked/sie-server:latest-cpu-default

# GPU (NVIDIA)
docker run --gpus all -p 8080:8080 ghcr.io/superlinked/sie-server:latest-cuda12-default
```

In [ ]:
!pip install -q sie-sdk

## 2. Encode

Generate dense embeddings with a 400M-parameter model. The model downloads on first use (~1-2 min), then subsequent calls are fast.

In [ ]:
from sie_sdk import SIEClient
from sie_sdk.types import Item

client = SIEClient("http://localhost:8080")

result = client.encode("NovaSearch/stella_en_400M_v5", Item(text="Hello world"))
print(f"Shape: {result['dense'].shape}")   # (1024,)
print(f"First 5 values: {result['dense'][:5]}")

## 3. Score

Rerank documents by relevance using a cross-encoder.

In [ ]:
scores = client.score(
    "BAAI/bge-reranker-v2-m3",
    Item(text="What is machine learning?"),
    [Item(text="ML learns from data."), Item(text="The weather is sunny.")]
)

for s in scores["scores"]:
    print(f"  rank {s['rank']}: {s['score']:.3f} - {s['item_id']}")

## 4. Extract

Zero-shot named entity recognition - no training data needed.

In [ ]:
result = client.extract(
    "urchade/gliner_multi-v2.1",
    Item(text="Tim Cook is the CEO of Apple."),
    labels=["person", "organization"]
)

for e in result["entities"]:
    print(f"  {e['label']}: {e['text']} ({e['score']:.2f})")

## Next steps

- [API reference](https://sie.dev/docs/reference/sdk) - full SDK documentation
- [All models](https://sie.dev/docs/reference/models) - 85+ supported models
- [Deployment guide](https://sie.dev/docs/deployment/docker) - run in production with Docker or Kubernetes
- [GitHub](https://github.com/superlinked/sie) - star the repo if this was useful